# Autoregressive models and the ACF / PACF: lab notebook

**Statistical Modelling and Forecasting, ARIMA and vector autoregression**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.tsa.stattools import acf, pacf, arma_order_select_ic
from statsmodels.regression.linear_model import yule_walker
from statsmodels.tsa.ar_model import AutoReg

### 1. Simulate an AR(2) process: y_t = 1.2 y_{t-1} - 0.6 y_{t-2} + e_t

In [ ]:
# 1. Simulate an AR(2) process: y_t = 1.2 y_{t-1} - 0.6 y_{t-2} + e_t
phi = [1.2, -0.6]
proc = ArmaProcess(np.r_[1, -np.array(phi)], [1])
print("stationary:", proc.isstationary, "| roots modulus:", np.round(np.abs(proc.arroots), 3))
y = proc.generate_sample(nsample=300, distrvs=np.random.default_rng(3).standard_normal)

### 2. Theoretical against sample ACF and PACF

In [ ]:
# 2. Theoretical against sample ACF and PACF
print("lag   theory ACF  sample ACF  sample PACF")
th = proc.acf(7)
sa = acf(y, nlags=6)
sp = pacf(y, nlags=6)
for k in range(1, 7):
    print(f"{k:3d}  {th[k]:10.3f}  {sa[k]:10.3f}  {sp[k]:11.3f}")
print("95% band: +-", round(1.96 / np.sqrt(len(y)), 3), "-> the PACF cuts off after lag 2")

### 3. Estimate by Yule-Walker and by least squares (AutoReg)

In [ ]:
# 3. Estimate by Yule-Walker and by least squares (AutoReg)
rho, sigma = yule_walker(y, order=2, method="mle")
print("Yule-Walker phi:", np.round(rho, 3), "| sigma:", round(sigma, 3))
ar = AutoReg(y, lags=2, trend="n").fit()
print("OLS (AutoReg) phi:", np.round(ar.params, 3))

### 4. Choose the order by AIC and forecast

In [ ]:
# 4. Choose the order by AIC and forecast
for p in range(1, 6):
    print("AR(%d) AIC = %.1f" % (p, AutoReg(y, lags=p, trend="n").fit().aic))
print("next 5 forecasts:", np.round(ar.predict(start=len(y), end=len(y) + 4), 3))

### 5. The numerical problem: AR(1) with c = 8, phi = 0.6, sigma^2 = 4, y_T = 25

In [ ]:
# 5. The numerical problem: AR(1) with c = 8, phi = 0.6, sigma^2 = 4, y_T = 25
c, ph, s2, yT = 8, 0.6, 4, 25
mu = c / (1 - ph)
print("mean:", mu, "| variance:", s2 / (1 - ph ** 2), "| rho_3:", ph ** 3, "| 2-step forecast:", mu + ph ** 2 * (yT - mu))
r1, r2 = 0.7, 0.3
p2 = (r2 - r1 ** 2) / (1 - r1 ** 2)
print("Yule-Walker AR(2): phi2 =", round(p2, 4), "phi1 =", round(r1 * (1 - p2), 4))

## Expected output

Running all cells above should print:

```
stationary: True | roots modulus: [1.291 1.291]
lag   theory ACF  sample ACF  sample PACF
  1       0.750       0.720        0.723
  2       0.300       0.242       -0.583
  3      -0.090      -0.160       -0.061
  4      -0.288      -0.331        0.042
  5      -0.292      -0.278        0.022
  6      -0.177      -0.097        0.045
95% band: +- 0.113 -> the PACF cuts off after lag 2
Yule-Walker phi: [ 1.135 -0.575] | sigma: 1.037
OLS (AutoReg) phi: [ 1.178 -0.607]
AR(1) AIC = 987.9
AR(2) AIC = 850.4
AR(3) AIC = 849.6
AR(4) AIC = 848.3
AR(5) AIC = 848.0
next 5 forecasts: [-3.382 -1.689  0.063  1.098  1.256]
mean: 20.0 | variance: 6.25 | rho_3: 0.21599999999999997 | 2-step forecast: 21.8
Yule-Walker AR(2): phi2 = -0.3725 phi1 = 0.9608
```

## Practice exercises

Try these in the cells below.

1. Set p = 1 and move φ from 0.9 to −0.9. Describe how the ACF and the series change. Where does the PACF stop?
2. Choose AR(2) with φ₁ = 1.2, φ₂ = −0.6. The ACF turns into a damped wave. Read the length of one cycle from the plot.
3. Push φ₁ + φ₂ above 1 and note how the simulator reacts. Why can the process not be simulated?

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
